# 📘 Notebook 11: Object-Oriented Programming
**Expected: ~8 questions out of 65** — One of the most tested topics!

---
# 📖 THEORY
---

In [ ]:
# Class basics
class Dog:
    species = "Canis familiaris"  # Class variable (shared)

    def __init__(self, name, age):
        self.name = name    # Instance variable (per object)
        self.age = age

    def bark(self):
        return f"{self.name} says Woof!"

    def __str__(self):       # User-friendly string
        return f"Dog({self.name}, {self.age})"

    def __repr__(self):      # Developer string
        return f"Dog('{self.name}', {self.age})"

d = Dog("Rex", 5)
print(d.bark())        # Rex says Woof!
print(d)               # Dog(Rex, 5) - uses __str__
print(repr(d))         # Dog('Rex', 5) - uses __repr__
print(d.species)       # Canis familiaris

## Instance vs Class Variables

In [ ]:
class MyClass:
    shared = []  # Class variable - shared by ALL instances!

    def __init__(self):
        self.own = []  # Instance variable - unique per instance

a = MyClass()
b = MyClass()

# Instance variables are independent
a.own.append(1)
print(a.own)    # [1]
print(b.own)    # [] - independent!

# Class variable is SHARED!
a.shared.append(1)
print(b.shared)  # [1] - affected!
print(MyClass.shared)  # [1]

## @classmethod, @staticmethod, @property

In [ ]:
class Circle:
    pi = 3.14159

    def __init__(self, radius):
        self._radius = radius

    @property
    def radius(self):
        return self._radius

    @radius.setter
    def radius(self, value):
        if value < 0:
            raise ValueError("Radius cannot be negative")
        self._radius = value

    @classmethod
    def from_diameter(cls, diameter):
        return cls(diameter / 2)

    @staticmethod
    def is_valid_radius(value):
        return value >= 0

c = Circle(5)
print(c.radius)     # 5 (property getter)
c.radius = 10       # property setter
print(c.radius)     # 10

c2 = Circle.from_diameter(20)  # classmethod
print(c2.radius)    # 10.0

print(Circle.is_valid_radius(5))   # True (staticmethod)

## Inheritance & MRO

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name
    def speak(self):
        return "..."

class Dog(Animal):
    def speak(self):
        return "Woof!"

class Cat(Animal):
    def speak(self):
        return "Meow!"

# Polymorphism
for animal in [Dog("Rex"), Cat("Whiskers")]:
    print(f"{animal.name}: {animal.speak()}")

print(isinstance(Dog("Rex"), Animal))  # True
print(issubclass(Dog, Animal))         # True

In [ ]:
# Multiple Inheritance & MRO (Method Resolution Order)
class A:
    def method(self):
        return "A"

class B(A):
    def method(self):
        return "B"

class C(A):
    def method(self):
        return "C"

class D(B, C):
    pass

d = D()
print(d.method())     # "B" - follows MRO
print(D.__mro__)      # D -> B -> C -> A -> object
print(D.mro())        # Same as above

## Encapsulation & Name Mangling

In [ ]:
class MyClass:
    def __init__(self):
        self.public = "public"         # Public
        self._protected = "protected"   # Protected (convention)
        self.__private = "private"      # Private (name mangling)

obj = MyClass()
print(obj.public)       # "public"
print(obj._protected)   # "protected" (accessible but shouldn't)
# print(obj.__private)  # AttributeError!
print(obj._MyClass__private)  # "private" (name mangling)

## Important Dunder Methods

In [ ]:
class Vector:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __add__(self, other):
        return Vector(self.x + other.x, self.y + other.y)

    def __eq__(self, other):
        return self.x == other.x and self.y == other.y

    def __len__(self):
        return 2

    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

    def __bool__(self):
        return self.x != 0 or self.y != 0

    def __getitem__(self, index):
        return (self.x, self.y)[index]

    def __call__(self):
        return (self.x, self.y)

v1 = Vector(1, 2)
v2 = Vector(3, 4)
print(v1 + v2)         # Vector(4, 6)
print(v1 == Vector(1,2))  # True
print(len(v1))         # 2
print(bool(Vector(0,0)))  # False
print(v1[0])           # 1
print(v1())            # (1, 2)

## Abstract Base Classes

In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self):
        pass

    @abstractmethod
    def perimeter(self):
        pass

# shape = Shape()  # TypeError! Can't instantiate abstract class

class Rectangle(Shape):
    def __init__(self, w, h):
        self.w, self.h = w, h

    def area(self):
        return self.w * self.h

    def perimeter(self):
        return 2 * (self.w + self.h)

r = Rectangle(3, 4)
print(r.area())       # 12
print(r.perimeter())  # 14

## dataclasses

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Point:
    x: float
    y: float
    label: str = "origin"

p = Point(3, 4)
print(p)              # Point(x=3, y=4, label='origin')
print(p == Point(3, 4))  # True (auto __eq__)

@dataclass(frozen=True)  # Immutable
class FrozenPoint:
    x: float
    y: float

fp = FrozenPoint(1, 2)
# fp.x = 3  # FrozenInstanceError!

## __slots__

In [ ]:
class WithSlots:
    __slots__ = ['x', 'y']
    def __init__(self, x, y):
        self.x, self.y = x, y

obj = WithSlots(1, 2)
# obj.z = 3  # AttributeError! No __dict__
# Saves memory, slightly faster attribute access

---
# 🔬 DEEP DIVE — Traps & Gotchas
---

In [ ]:
# TRAP 1: Mutable class variable
class Team:
    members = []  # Shared by ALL instances!

t1 = Team()
t2 = Team()
t1.members.append("Alice")
print(t2.members)  # ['Alice'] — affected!
print(Team.members) # ['Alice'] — same list!

# FIX: Use instance variable in __init__
class TeamFixed:
    def __init__(self):
        self.members = []  # Each instance gets its own

In [ ]:
# TRAP 2: __new__ vs __init__
class Singleton:
    _instance = None
    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance
    def __init__(self):
        self.x = 42

a = Singleton()
b = Singleton()
print(a is b)  # True — same instance!
print(id(a) == id(b))  # True

In [ ]:
# TRAP 3: super() in multiple inheritance
class A:
    def __init__(self):
        print("A", end=" ")
        super().__init__()

class B(A):
    def __init__(self):
        print("B", end=" ")
        super().__init__()

class C(A):
    def __init__(self):
        print("C", end=" ")
        super().__init__()

class D(B, C):
    def __init__(self):
        print("D", end=" ")
        super().__init__()

d = D()  # D B C A — follows MRO!
print()
print(D.__mro__)

In [ ]:
# TRAP 4: __eq__ and __hash__
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y
    def __eq__(self, other):
        return self.x == other.x and self.y == other.y

p = Point(1, 2)
# {p}  # TypeError: unhashable type
# If you override __eq__, __hash__ becomes None!
# FIX: Also define __hash__
class PointFixed:
    def __init__(self, x, y):
        self.x, self.y = x, y
    def __eq__(self, other):
        return self.x == other.x and self.y == other.y
    def __hash__(self):
        return hash((self.x, self.y))

p = PointFixed(1, 2)
print({p})  # Works!

---
# 🧪 INTERACTIVE EXAMPLES
---

In [ ]:
# What will this code print?
class A:
    def greet(self):
        return "Hello from A"

class B(A):
    pass

class C(A):
    def greet(self):
        return "Hello from C"

class D(B, C):
    pass

# What does D().greet() return?

In [ ]:
print(D().greet())  # "Hello from C" — MRO: D->B->C->A
print(D.__mro__)

In [ ]:
# What will this code print?
class Counter:
    count = 0
    def __init__(self):
        self.__class__.count += 1
    def __del__(self):
        self.__class__.count -= 1

a = Counter()
b = Counter()
print(Counter.count)  # ?

In [ ]:
# Answer: 2 — two instances created, count incremented twice

In [ ]:
# What will this code print?
class MyClass:
    def __init__(self, x):
        self.x = x
    def __repr__(self):
        return f"MyClass({self.x})"
    def __str__(self):
        return f"Value: {self.x}"

obj = MyClass(42)
print(obj)        # ?
print([obj])      # ?
print(repr(obj))  # ?

In [ ]:
# Answer:
# print(obj) -> "Value: 42" (uses __str__)
# print([obj]) -> "[MyClass(42)]" (list uses __repr__ for elements!)
# print(repr(obj)) -> "MyClass(42)" (uses __repr__)

---
# 💻 EXERCISES
---

In [ ]:
# Exercise 1: Create a class BankAccount with deposit, withdraw, balance
class BankAccount:
    def __init__(self, balance=0):
        # YOUR CODE HERE
        pass
    
    def deposit(self, amount):
        # YOUR CODE HERE
        pass
    
    def withdraw(self, amount):
        # YOUR CODE HERE (raise ValueError if insufficient funds)
        pass
    
    @property
    def balance(self):
        # YOUR CODE HERE
        pass

# Tests
acc = BankAccount(100)
assert acc.balance == 100
acc.deposit(50)
assert acc.balance == 150
acc.withdraw(30)
assert acc.balance == 120
try:
    acc.withdraw(200)
    assert False, "Should raise ValueError"
except ValueError:
    pass
print("\u2705 Exercise 1 passed!")

In [ ]:
# Solution 1
class BankAccount:
    def __init__(self, balance=0):
        self._balance = balance
    def deposit(self, amount):
        self._balance += amount
    def withdraw(self, amount):
        if amount > self._balance:
            raise ValueError("Insufficient funds")
        self._balance -= amount
    @property
    def balance(self):
        return self._balance

acc = BankAccount(100)
assert acc.balance == 100
acc.deposit(50)
assert acc.balance == 150
acc.withdraw(30)
assert acc.balance == 120
try:
    acc.withdraw(200)
    assert False
except ValueError:
    pass
print("\u2705 Exercise 1 passed!")

In [ ]:
# Exercise 2: Create a class hierarchy Shape -> Circle, Rectangle with area()
from abc import ABC, abstractmethod
import math

class Shape(ABC):
    @abstractmethod
    def area(self):
        pass

class Circle(Shape):
    def __init__(self, radius):
        # YOUR CODE HERE
        pass
    def area(self):
        # YOUR CODE HERE
        pass

class Rectangle(Shape):
    def __init__(self, width, height):
        # YOUR CODE HERE
        pass
    def area(self):
        # YOUR CODE HERE
        pass

# Tests
c = Circle(5)
assert abs(c.area() - 78.5398) < 0.01
r = Rectangle(3, 4)
assert r.area() == 12
print("\u2705 Exercise 2 passed!")

In [ ]:
# Solution 2
from abc import ABC, abstractmethod
import math

class Shape(ABC):
    @abstractmethod
    def area(self):
        pass

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius
    def area(self):
        return math.pi * self.radius ** 2

class Rectangle(Shape):
    def __init__(self, width, height):
        self.width, self.height = width, height
    def area(self):
        return self.width * self.height

c = Circle(5)
assert abs(c.area() - 78.5398) < 0.01
r = Rectangle(3, 4)
assert r.area() == 12
print("\u2705 Exercise 2 passed!")

In [ ]:
# Exercise 3: Implement __add__, __eq__, __repr__ for a Vector class
class Vector:
    def __init__(self, x, y):
        # YOUR CODE HERE
        pass
    def __add__(self, other):
        # YOUR CODE HERE
        pass
    def __eq__(self, other):
        # YOUR CODE HERE
        pass
    def __repr__(self):
        # YOUR CODE HERE
        pass

# Tests
v1 = Vector(1, 2)
v2 = Vector(3, 4)
v3 = v1 + v2
assert v3 == Vector(4, 6)
assert repr(v1) == "Vector(1, 2)"
print("\u2705 Exercise 3 passed!")

In [ ]:
# Solution 3
class Vector:
    def __init__(self, x, y):
        self.x, self.y = x, y
    def __add__(self, other):
        return Vector(self.x + other.x, self.y + other.y)
    def __eq__(self, other):
        return self.x == other.x and self.y == other.y
    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

v1 = Vector(1, 2)
v2 = Vector(3, 4)
v3 = v1 + v2
assert v3 == Vector(4, 6)
assert repr(v1) == "Vector(1, 2)"
print("\u2705 Exercise 3 passed!")

In [ ]:
# Exercise 4: Create a Stack class using __len__, __bool__, __repr__
class Stack:
    def __init__(self):
        # YOUR CODE HERE
        pass
    def push(self, item):
        # YOUR CODE HERE
        pass
    def pop(self):
        # YOUR CODE HERE
        pass
    def peek(self):
        # YOUR CODE HERE
        pass
    def __len__(self):
        # YOUR CODE HERE
        pass
    def __bool__(self):
        # YOUR CODE HERE
        pass
    def __repr__(self):
        # YOUR CODE HERE
        pass

# Tests
s = Stack()
assert not s  # Empty stack is falsy
s.push(1)
s.push(2)
s.push(3)
assert len(s) == 3
assert s.peek() == 3
assert s.pop() == 3
assert len(s) == 2
assert bool(s) == True
print("\u2705 Exercise 4 passed!")

In [ ]:
# Solution 4
class Stack:
    def __init__(self):
        self._items = []
    def push(self, item):
        self._items.append(item)
    def pop(self):
        return self._items.pop()
    def peek(self):
        return self._items[-1]
    def __len__(self):
        return len(self._items)
    def __bool__(self):
        return len(self._items) > 0
    def __repr__(self):
        return f"Stack({self._items})"

s = Stack()
assert not s
s.push(1); s.push(2); s.push(3)
assert len(s) == 3
assert s.peek() == 3
assert s.pop() == 3
assert len(s) == 2
assert bool(s) == True
print("\u2705 Exercise 4 passed!")

In [ ]:
# Exercise 5: Create a @dataclass Person with name, age, and a method is_adult()
from dataclasses import dataclass

# YOUR CODE HERE: create Person dataclass

# Tests
# p = Person("Alice", 25)
# assert p.is_adult() == True
# assert str(p) == "Person(name='Alice', age=25)"
# p2 = Person("Alice", 25)
# assert p == p2
# print("\u2705 Exercise 5 passed!")

In [ ]:
# Solution 5
from dataclasses import dataclass

@dataclass
class Person:
    name: str
    age: int
    
    def is_adult(self):
        return self.age >= 18

p = Person("Alice", 25)
assert p.is_adult() == True
assert str(p) == "Person(name='Alice', age=25)"
p2 = Person("Alice", 25)
assert p == p2
print("\u2705 Exercise 5 passed!")

---
# ⚡ SPEED ROUND (17 seconds each)
---

In [ ]:
# \u26a1 Q1: self refers to?

In [ ]:
print("\u2705 Instance")

In [ ]:
# \u26a1 Q2: @classmethod gets?

In [ ]:
print("\u2705 cls")

In [ ]:
# \u26a1 Q3: @staticmethod gets?

In [ ]:
print("\u2705 Nothing")

In [ ]:
# \u26a1 Q4: MRO order for D(B,C)?

In [ ]:
print("\u2705 D->B->C->parent->object")

In [ ]:
# \u26a1 Q5: __str__ vs __repr__: print uses?

In [ ]:
print("\u2705 __str__")

In [ ]:
# \u26a1 Q6: Can instantiate ABC?

In [ ]:
print("\u2705 No!")

In [ ]:
# \u26a1 Q7: __slots__ prevents?

In [ ]:
print("\u2705 __dict__ creation")

In [ ]:
# \u26a1 Q8: Name mangling: __x becomes?

In [ ]:
print("\u2705 _ClassName__x")

In [ ]:
# \u26a1 Q9: super() calls?

In [ ]:
print("\u2705 Parent class method")

In [ ]:
# \u26a1 Q10: If __eq__ overridden, __hash__=?

In [ ]:
print("\u2705 None")

---
# 📝 QUIZ (25 Questions)
---

In [ ]:
# Q1 🟡: MRO of D(B,C) where B(A),C(A)?
# A) D-B-C-A  B) D-B-A-C  C) D-C-B-A

In [ ]:
print("✅ A) D-B-C-A")
print("📖 C3 linearization: D->B->C->A->object")

In [ ]:
# Q2 🟡: print() calls __str__ or __repr__?
# A) __str__  B) __repr__

In [ ]:
print("✅ A) __str__")
print("📖 print uses __str__, falling back to __repr__")

In [ ]:
# Q3 🔴: Name mangling: obj.__x becomes?
# A) obj.__x  B) obj._ClassName__x

In [ ]:
print("✅ B) obj._ClassName__x")
print("📖 Python mangles __ prefix")

In [ ]:
# Q4 🟡: Mutable class variable trap?
# A) Each instance has own  B) All share same

In [ ]:
print("✅ B) All share same")
print("📖 Mutable class vars are shared references")

In [ ]:
# Q5 🟢: self refers to?
# A) Class  B) Instance  C) Module

In [ ]:
print("✅ B) Instance")
print("📖 self is the instance")

In [ ]:
# Q6 🟡: @classmethod receives?
# A) self  B) cls  C) Nothing

In [ ]:
print("✅ B) cls")
print("📖 classmethod gets class as first arg")

In [ ]:
# Q7 🟡: @staticmethod receives?
# A) self  B) cls  C) Nothing

In [ ]:
print("✅ C) Nothing")
print("📖 No self or cls")

In [ ]:
# Q8 🟡: Can instantiate ABC?
# A) Yes  B) No

In [ ]:
print("✅ B) No")
print("📖 Abstract class cannot be instantiated")

In [ ]:
# Q9 🟡: __init__ return value?
# A) None  B) self  C) Object

In [ ]:
print("✅ A) None")
print("📖 __init__ must return None")

In [ ]:
# Q10 🟡: isinstance(True, int)?
# A) True  B) False

In [ ]:
print("✅ A) True")
print("📖 bool is subclass of int")

In [ ]:
# Q11 🟡: __eq__ overridden -> __hash__?
# A) Unchanged  B) Set to None

In [ ]:
print("✅ B) Set to None")
print("📖 Must implement __hash__ if __eq__ overridden")

In [ ]:
# Q12 🟡: If no __bool__, Python checks?
# A) __len__  B) __eq__  C) Always True

In [ ]:
print("✅ A) __len__")
print("📖 Falls back to __len__, then True")

In [ ]:
# Q13 🟢: super() does?
# A) Calls parent  B) Creates new class

In [ ]:
print("✅ A) Calls parent")
print("📖 Delegates to parent class")

In [ ]:
# Q14 🟡: @property makes?
# A) Class variable  B) Getter method as attribute

In [ ]:
print("✅ B)")
print("📖 Access method like attribute")

In [ ]:
# Q15 🟡: __new__ vs __init__?
# A) Same  B) __new__ creates, __init__ initializes

In [ ]:
print("✅ B)")
print("📖 __new__ creates instance, __init__ configures it")

In [ ]:
# Q16 🟢: Multiple inheritance Python supports?
# A) Yes  B) No

In [ ]:
print("✅ A) Yes")
print("📖 Python supports multiple inheritance")

In [ ]:
# Q17 🟡: @dataclass auto-generates?
# A) __init__ only  B) __init__, __repr__, __eq__

In [ ]:
print("✅ B)")
print("📖 Auto-generates several dunder methods")

In [ ]:
# Q18 🟡: __slots__ saves memory by?
# A) No __dict__  B) No methods

In [ ]:
print("✅ A) No __dict__")
print("📖 Prevents dynamic attribute creation")

In [ ]:
# Q19 🔴: d={True:'a',1:'b',1.0:'c'} relates to OOP how?
# A) Inheritance  B) __hash__ and __eq__

In [ ]:
print("✅ B)")
print("📖 True==1==1.0 due to __eq__, same hash")

In [ ]:
# Q20 🟡: __str__ not defined, print uses?
# A) __repr__  B) Error  C) Memory address

In [ ]:
print("✅ A) __repr__")
print("📖 Falls back to __repr__")

In [ ]:
# Q21 🟡: Composition vs Inheritance?
# A) Same  B) has-a vs is-a

In [ ]:
print("✅ B)")
print("📖 Composition=has-a, Inheritance=is-a")

In [ ]:
# Q22 🟢: All classes inherit from?
# A) type  B) object  C) None

In [ ]:
print("✅ B) object")
print("📖 object is the root")

In [ ]:
# Q23 🟡: type(MyClass) is?
# A) object  B) type  C) class

In [ ]:
print("✅ B) type")
print("📖 type is the metaclass")

In [ ]:
# Q24 🟡: frozen=True in dataclass means?
# A) Thread-safe  B) Immutable

In [ ]:
print("✅ B) Immutable")
print("📖 Cannot modify attributes after creation")

In [ ]:
# Q25 🔴: Diamond problem solved by?
# A) First match  B) MRO (C3)

In [ ]:
print("✅ B) MRO (C3)")
print("📖 C3 linearization algorithm")

---
# 🔑 KEY TAKEAWAYS
---
1. **MRO (Method Resolution Order)** uses C3 linearization — check with `__mro__`
2. **`__str__` vs `__repr__`** — print() calls __str__ first, falls back to __repr__
3. **Name mangling**: `__x` becomes `_ClassName__x`
4. **Mutable class variables are shared** — use instance variables instead
5. **`@classmethod`** gets `cls`, **`@staticmethod`** gets nothing
6. **ABC cannot be instantiated** — must implement all abstract methods
7. **If `__eq__` is overridden, `__hash__` becomes None** — implement both
8. **`__slots__`** prevents `__dict__`, saves memory